# Multi-Task Sequential Recommendation — Results Comparison

This notebook:
1. Loads all MLflow runs from the `multitask_seqrec` experiment
2. Displays a comparison table across all three models
3. Plots training loss curves and final metric comparisons

Run cells top-to-bottom after completing all three training scripts.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

from config import MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()
print(f'Tracking URI: {MLFLOW_TRACKING_URI}')

## 1. Load experiment runs

In [ ]:
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if experiment is None:
    raise RuntimeError(
        f"Experiment '{MLFLOW_EXPERIMENT}' not found. "
        "Run the training scripts first."
    )

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['attributes.start_time DESC'],
)
print(f"Found {len(runs)} runs in experiment '{MLFLOW_EXPERIMENT}'")
for r in runs:
    print(f"  {r.data.tags.get('mlflow.runName', r.info.run_id):20s}  "
          f"status={r.info.status}")

## 2. Final Metrics Comparison Table

In [ ]:
MODEL_ORDER = ['sasrec', 'shared_bottom', 'mmoe']
MODEL_LABELS = {
    'sasrec':         'SASRec (single-task)',
    'shared_bottom':  'SASRec + Shared-Bottom',
    'mmoe':           'SASRec + MMoE',
}

# Pick the most recent run for each model name
best_runs = {}
for run in runs:
    name = run.data.tags.get('mlflow.runName', '')
    if name in MODEL_ORDER and name not in best_runs:
        best_runs[name] = run

# Build comparison DataFrame
rows = []
for model_key in MODEL_ORDER:
    if model_key not in best_runs:
        print(f'  WARNING: no run found for {model_key}')
        continue
    m = best_runs[model_key].data.metrics
    p = best_runs[model_key].data.params
    row = {
        'Model': MODEL_LABELS[model_key],
        'NDCG@10': m.get('test_NDCG_at_10', float('nan')),
        'HR@10':   m.get('test_HR_at_10',   float('nan')),
        'MAE':     m.get('test_MAE',         float('nan')),
        '#Params': int(p.get('params', 0)),
    }
    rows.append(row)

df_results = pd.DataFrame(rows).set_index('Model')

# Format
styled = (
    df_results.style
    .format({'NDCG@10': '{:.4f}', 'HR@10': '{:.4f}',
             'MAE': '{:.4f}', '#Params': '{:,}'})
    .background_gradient(subset=['NDCG@10', 'HR@10'], cmap='Greens')
    .background_gradient(subset=['MAE'], cmap='RdYlGn_r')
    .set_caption('Test-set Results Comparison')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '14px'), ('font-weight', 'bold')]
    }])
)
display(styled)
print('\nRaw values:')
print(df_results.to_string())

## 3. Training Loss Curves

In [ ]:
def get_metric_history(run, metric_name):
    """Fetch per-step metric history for a run."""
    history = client.get_metric_history(run.info.run_id, metric_name)
    if not history:
        return [], []
    steps = [h.step for h in history]
    values = [h.value for h in history]
    return steps, values


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = {'sasrec': 'C0', 'shared_bottom': 'C1', 'mmoe': 'C2'}

# --- Training loss ---
ax = axes[0]
for key in MODEL_ORDER:
    if key not in best_runs:
        continue
    metric = 'train_click_loss' if key != 'sasrec' else 'train_loss'
    steps, vals = get_metric_history(best_runs[key], metric)
    ax.plot(steps, vals, label=MODEL_LABELS[key], color=colors[key])
ax.set_xlabel('Epoch')
ax.set_ylabel('Click Loss')
ax.set_title('Training Click Loss')
ax.legend(fontsize=9)

# --- Val NDCG@10 ---
ax = axes[1]
for key in MODEL_ORDER:
    if key not in best_runs:
        continue
    steps, vals = get_metric_history(best_runs[key], 'val_NDCG_at_10')
    ax.plot(steps, vals, label=MODEL_LABELS[key], color=colors[key])
ax.set_xlabel('Epoch')
ax.set_ylabel('NDCG@10')
ax.set_title('Validation NDCG@10')
ax.legend(fontsize=9)

# --- Val HR@10 ---
ax = axes[2]
for key in MODEL_ORDER:
    if key not in best_runs:
        continue
    steps, vals = get_metric_history(best_runs[key], 'val_HR_at_10')
    ax.plot(steps, vals, label=MODEL_LABELS[key], color=colors[key])
ax.set_xlabel('Epoch')
ax.set_ylabel('HR@10')
ax.set_title('Validation HR@10')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../notebooks/training_curves.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/training_curves.png')

## 4. Final Metric Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

model_names = df_results.index.tolist()
x = np.arange(len(model_names))
bar_colors = [colors.get(k, 'steelblue') for k in MODEL_ORDER if MODEL_LABELS.get(k) in model_names]

metrics_plot = [
    ('NDCG@10',  'NDCG@10 (higher is better)',  'Greens'),
    ('HR@10',    'HR@10 (higher is better)',    'Blues'),
    ('MAE',      'MAE (lower is better)',        'Oranges'),
]

for ax, (col, title, cmap) in zip(axes, metrics_plot):
    vals = df_results[col].values
    if np.all(np.isnan(vals)):
        ax.set_title(f'{col} — no data')
        continue
    bars = ax.bar(x, vals, color=sns.color_palette(cmap, len(x))[::-1],
                  edgecolor='grey', linewidth=0.5)
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [n.replace('SASRec + ', 'SASRec +\n') for n in model_names],
        fontsize=9
    )
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, vals[~np.isnan(vals)].max() * 1.18)

plt.tight_layout()
plt.savefig('../notebooks/metric_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: notebooks/metric_comparison.png')

## 5. Rating Prediction History (MTL models only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, key, title in [
    (axes[0], 'shared_bottom', 'Shared-Bottom: Rating Loss'),
    (axes[1], 'mmoe',          'MMoE: Rating Loss'),
]:
    if key not in best_runs:
        ax.set_title(f'{title} — no data')
        continue
    steps, vals = get_metric_history(best_runs[key], 'train_rating_loss')
    ax.plot(steps, vals, color=colors[key])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Rating MSE Loss')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('../notebooks/rating_loss.png', bbox_inches='tight')
plt.show()

## 6. Serving Layer Smoke Test

Make sure the FastAPI server is running first:
```bash
uvicorn serve.app:app --host 0.0.0.0 --port 8000
```

In [ ]:
import requests
import pickle, os
from config import PROCESSED_DATA_PATH

BASE_URL = 'http://localhost:8000'

# Health check
try:
    resp = requests.get(f'{BASE_URL}/health', timeout=5)
    print('Health:', resp.json())
except Exception as e:
    print(f'Server not reachable: {e}')
    print('Start with: uvicorn serve.app:app --host 0.0.0.0 --port 8000')

In [ ]:
# Load a real user sequence for testing
data_path = os.path.join(PROCESSED_DATA_PATH, 'data.pkl')
vocab_path = os.path.join(PROCESSED_DATA_PATH, 'vocab.pkl')

with open(data_path, 'rb') as f:
    data = pickle.load(f)
with open(vocab_path, 'rb') as f:
    vocab = pickle.load(f)

id2item = vocab['id2item']

# Pick the first test user
uid, (ctx, tgt) = next(iter(data['test_seqs'].items()))
ctx_items_str = [id2item[i] for i, _ in ctx[-10:]]  # last 10 interactions
target_item_str = id2item[tgt[0]]

print(f'User: {uid}')
print(f'Context (last 10): {ctx_items_str}')
print(f'Target item:       {target_item_str}')

# Recommend
resp = requests.post(
    f'{BASE_URL}/recommend',
    json={'item_ids': ctx_items_str, 'top_k': 10},
    timeout=30,
)
resp_json = resp.json()
print(f'\nLatency: {resp_json["latency_ms"]:.1f}ms')
print(f'Top-10 recommendations:')
recs = pd.DataFrame(resp_json['recommendations'])
display(recs)

# Check if target is in top-10
top10_ids = [r['item_id'] for r in resp_json['recommendations']]
hit = target_item_str in top10_ids
print(f'\nTarget in top-10: {hit}')

## 7. Summary

The table and plots above summarise the benchmark results.

**Key findings** (fill in after running experiments):
- MMoE vs SASRec: shows how multi-task learning affects ranking quality
- Shared-Bottom vs MMoE: demonstrates the value of expert specialization via gating
- Rating MAE: only available for MTL models (SharedBottom and MMoE)

All runs are tracked in MLflow. Start the UI with:
```bash
mlflow ui --backend-store-uri mlruns/
```